# Fetch external data for the 3-wheel essentiality system

Two parallel paths, one click:
1. **feba.db** — Fitness Browser per-condition RB-TnSeq fitness (~7 GB SQLite). Directly attacks the 25% soup / 30% essential-recall ceiling — *conditional* essentiality across hundreds of media.
2. **BiGG metabolic models** — pulls ~15 genome-scale models so the FBA metabolic wheel covers ~20% of our corpus instead of 5%.

Both saved to **Drive → `cell_external_data/`** so the sandbox can read them back via the Drive connector.

**Runtime: any CPU instance, no GPU needed. ~15 min total.**

In [ ]:
# 0. mount drive
from google.colab import drive
drive.mount('/content/drive')
DST = '/content/drive/MyDrive/cell_external_data'
import os, time, hashlib
os.makedirs(DST, exist_ok=True)
os.makedirs(f'{DST}/bigg_models', exist_ok=True)
print('destination:', DST)

## Path 1 — feba.db (Fitness Browser conditional fitness)

In [ ]:
# 1. download feba.db -- the canonical Fitness Browser SQLite
# Mirror locations to try in order (each is the official Arkin-lab fitness data):
FEBA_URLS = [
    'https://fit.genomics.lbl.gov/cgi_data/feba.db',
    'http://genomics.lbl.gov/supplemental/bigfit/feba.db',
]
feba_path = f'{DST}/feba.db'
if os.path.exists(feba_path) and os.path.getsize(feba_path) > 1e9:
    print(f'feba.db already in Drive ({os.path.getsize(feba_path)/1e9:.1f} GB) - skip')
else:
    for url in FEBA_URLS:
        print(f'trying {url} ...')
        rc = os.system(f'wget -q -c --timeout=30 -O "{feba_path}.tmp" "{url}"')
        if rc == 0 and os.path.getsize(f'{feba_path}.tmp') > 1e9:
            os.rename(f'{feba_path}.tmp', feba_path)
            print(f'  OK ({os.path.getsize(feba_path)/1e9:.1f} GB)')
            break
        else:
            os.system(f'rm -f "{feba_path}.tmp"')
            print('  failed, trying next mirror')
# verify it's a real SQLite
import sqlite3
con = sqlite3.connect(feba_path)
tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
print(f'\nfeba.db tables ({len(tables)}): {tables[:12]}...')
# quick orgs sample
try:
    orgs = [r[0] for r in con.execute('SELECT DISTINCT orgId FROM Organism LIMIT 20')]
    print('sample orgs:', orgs[:8])
except Exception as e:
    print('orgs query:', e)
con.close()

## Path 2 — BiGG metabolic models

Targeted at our highest-essential-count organisms (Pseudomonas spp., Ralstonia, Burkholderia, Sinorhizobium, S. aureus, mtub, Bacteroides, Desulfovibrio, Acinetobacter, Caulobacter).

In [ ]:
# 2. download BiGG models (small SBML files, 200KB - 5MB each)
# (organism in our corpus, BiGG model id, BiGG name, fmt)
BIGG_TARGETS = [
    ('beril_Putida',              'iJN1463', 'P. putida KT2440'),
    ('beril_Keio',                'iJO1366', 'E. coli K-12 MG1655'),    # already bundled but pull to confirm
    ('beril_Keio',                'iML1515', 'E. coli K-12 (newer)'),
    ('mtub',                      'iEK1011', 'M. tuberculosis H37Rv'),
    ('saur_NCTC8325_biotradis',   'iSB619',  'S. aureus N315'),
    ('beril_Btheta',              'iAH991',  'B. thetaiotaomicron VPI-5482'),
    ('beril_DvH',                 'iJF744',  'D. vulgaris Hildenborough'),
    ('beril_Smeli',               'iGD1575', 'S. meliloti 1021'),
    ('beril_Caulo',               'iRR1083', 'C. crescentus NA1000'),
    ('aeromonas',                 'iCB925',  'A. hydrophila (close)'),
    ('beril_BFirm',               'iWZ663',  'B. cenocepacia (close)'),
    ('beril_RalstoniaGMI1000',    'RsolanacearumSCRIPub00_S3',  'R. solanacearum (community)'),
    ('beril_pseudo1_N1B4',        'iPau21',  'P. aeruginosa PAO1 (close)'),
    ('beril_SyringaeB728a',       'iSyp1145', 'P. syringae B728a (close)'),
    ('beril_Methanococcus_S2',    'iAF692',  'Methanosarcina (close, archaea)'),
]
import urllib.request
manifest = []
for org, mid, name in BIGG_TARGETS:
    for ext in ('.xml.gz', '.json', '.xml'):
        url = f'http://bigg.ucsd.edu/static/models/{mid}{ext}'
        path = f'{DST}/bigg_models/{mid}{ext}'
        if os.path.exists(path) and os.path.getsize(path) > 1000:
            print(f'  {mid}: cached'); manifest.append((org, mid, name, path)); break
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            data = urllib.request.urlopen(req, timeout=30).read()
            if len(data) > 1000:
                open(path, 'wb').write(data)
                size_kb = len(data)/1024
                print(f'  {mid}: {size_kb:.0f} KB <- {name}')
                manifest.append((org, mid, name, path)); break
        except Exception as e:
            continue
    else:
        print(f'  {mid}: FAILED -- {name}')
print(f'\n{len(manifest)}/{len(BIGG_TARGETS)} models downloaded')

In [ ]:
# 3. write the manifest CSV (org -> model file) -- the sandbox uses this to wire FBA
import csv
with open(f'{DST}/bigg_manifest.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['our_organism', 'bigg_id', 'bigg_name', 'file_basename'])
    for org, mid, name, path in manifest:
        w.writerow([org, mid, name, os.path.basename(path)])
print('wrote bigg_manifest.csv')
!cat "{DST}/bigg_manifest.csv"

In [ ]:
# 4. validate each model loads + runs FBA (so we know they're usable downstream)
!pip -q install cobra >/dev/null 2>&1
import cobra, cobra.io, glob
summary = []
for org, mid, name, path in manifest:
    try:
        if path.endswith('.json'):
            m = cobra.io.load_json_model(path)
        else:
            m = cobra.io.read_sbml_model(path)
        g = m.slim_optimize()
        summary.append(dict(org=org, mid=mid, rxns=len(m.reactions), mets=len(m.metabolites), genes=len(m.genes), growth=round(float(g) if g else 0,3)))
        print(f'  {mid}: {len(m.reactions)} rxns | {len(m.genes)} genes | growth={g:.3f}/h')
    except Exception as e:
        print(f'  {mid}: load FAIL {type(e).__name__}: {str(e)[:60]}')
        summary.append(dict(org=org, mid=mid, error=str(e)[:80]))
import json
json.dump(summary, open(f'{DST}/bigg_validation.json','w'), indent=2)

In [ ]:
# 5. final inventory
import os
print('=== Drive contents ===')
for root, dirs, files in os.walk(DST):
    for f in files:
        p = os.path.join(root, f)
        print(f'  {os.path.getsize(p)/1024:>9.1f} KB  {p.replace(DST+"/","")}')
print(f'\nTotal size:', sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(DST) for f in fs)/1e9, 'GB')